# 04 — Evaluation & Analysis (Stage 4)

Evaluates each condition on GSM8K-test + MATH-200 (greedy), then aggregates accuracy with bootstrap CIs. Predictions are written to `MATHDISTILL_HOME/results/predictions/`.

In [ ]:
%cd /content/math-distillation
import os
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["MATHDISTILL_HOME"] = "/content/drive/MyDrive/math-distillation"
    from huggingface_hub import login; login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("bootstrap note:", e)

In [ ]:
# Run the conditions you have trained. Comment out what you don't need yet.
STUDENT = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER = "meta-llama/Llama-3.1-8B-Instruct"

# (i) zero-shot baseline
!python scripts/run_eval.py --model-id $STUDENT --name zeroshot
# (iii) teacher_1, seed 0
!python scripts/run_eval.py --model-id $STUDENT --adapter adapters/teacher_1/seed0 --name teacher_1_seed0
# (iv) teacher_3, seed 0
!python scripts/run_eval.py --model-id $STUDENT --adapter adapters/teacher_3/seed0 --name teacher_3_seed0
# (ii) gold, seed 0
!python scripts/run_eval.py --model-id $STUDENT --adapter adapters/gold/seed0 --name gold_seed0
# (v) teacher upper bound
!python scripts/run_eval.py --model-id $TEACHER --name teacher

In [ ]:
# Aggregate: accuracy + 95% bootstrap CI per prediction file.
import glob, os
import pandas as pd
from mathdistill.utils import read_jsonl
from mathdistill.metrics import accuracy, bootstrap_ci

pred_dir = os.path.join(os.environ["MATHDISTILL_HOME"], "results", "predictions")
rows = []
for path in sorted(glob.glob(os.path.join(pred_dir, "*.jsonl"))):
    data = read_jsonl(path)
    correct = [r["correct"] for r in data]
    lo, hi = bootstrap_ci(correct)
    rows.append({"file": os.path.basename(path), "n": len(correct),
                 "acc": round(accuracy(correct), 4),
                 "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})
df = pd.DataFrame(rows)
df